# 01 - Exploring the preprocessed LAML data

**Goal:** look at the data before any modeling. Nothing in this notebook changes the data, and nothing uses survival to choose features.

How to use it: *Kernel -> Restart and run all*, top to bottom. Figures are saved to `results/figures/`.

What we are checking:
1. **Survival:** does the outcome look like AML (short survival, steep early drop)?
2. **mRNA:** value distribution, barely-expressed genes, outlier patients.
3. **Methylation:** the expected beta-value shape, and why M-values may help.
4. **PCA per layer:** are there outlier patients, or a technical split, before we trust a model?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Settings ----
RULE = "drop"                      # "drop" = primary analysis, "floor" = sensitivity analysis
DATA = "../data/processed/"        # this notebook is in notebooks/, one level below the project root
FIG = "../results/figures/"
TOP_N = 5000                       # most variable features used for PCA
SEED = 42

mrna = pd.read_pickle(DATA + f"mrna_{RULE}.pkl")                    # genes  x patients
meth = pd.read_pickle(DATA + f"methylation_filtered_{RULE}.pkl")    # probes x patients
surv = pd.read_csv(DATA + f"survival_{RULE}.csv")                   # PatientID, Survival, Death

# All three tables must hold the same patients in the same order.
assert list(mrna.columns) == list(meth.columns) == list(surv["PatientID"])

print("rule:", RULE)
print("mRNA genes x patients:       ", mrna.shape)
print("methylation probes x patients:", meth.shape)
print("survival rows:                ", surv.shape)

## 1. Survival

`Survival` is time in days, and `Death` is 1 if the patient died, 0 if they were still alive at last follow-up (**censored**: we only know they survived *at least* that long).

A **Kaplan-Meier curve** estimates the fraction of patients still alive over time while handling censored patients correctly. At each death time: survival is multiplied by (1 - deaths / patients still at risk). Censored patients leave the "at risk" group without counting as deaths.

In [ ]:
from lifelines import KaplanMeierFitter

# 1. Initialize the fitter
kmf = KaplanMeierFitter()

# 2. Fit the model using your survival time and death event columns
kmf.fit(durations=surv["Survival"], event_observed=surv["Death"])

# 3. Get the median survival time directly
median_survival = kmf.median_survival_time_
print(f"Median Survival Time: {median_survival} days")

# 4. (Bonus) You can plot the curve instantly with built-in styling!
kmf.plot_survival_function()

In [ ]:
def kaplan_meier(time, event):
    """Return the event times and the survival probability after each of them."""
    time = np.asarray(time)
    event = np.asarray(event)
    order = np.argsort(time)
    time, event = time[order], event[order]

    n_at_risk = len(time)
    prob = 1.0 
    times, probs = [0], [1.0] #probability of survival = 1 at time 0
    for t in np.unique(time):
        at_t = time == t
        deaths = event[at_t].sum()
        if deaths > 0:
            prob *= 1 - deaths / n_at_risk
            times.append(t)
            probs.append(prob)
        n_at_risk -= at_t.sum()        # everyone with this time leaves the risk set
    return np.array(times), np.array(probs)


t, p = kaplan_meier(surv["Survival"], surv["Death"])
below = t[p <= 0.5]
median_survival = below[0] if len(below) else None

print("patients:", len(surv), "| deaths:", int(surv["Death"].sum()),
      "| censored:", int((surv["Death"] == 0).sum()))
print("median survival from the KM curve (days):", median_survival)
print("longest follow-up (days):", int(surv["Survival"].max()))

# censored patients are drawn as small ticks on the curve
cens = surv.loc[surv["Death"] == 0, "Survival"].to_numpy()
cens_prob = p[np.searchsorted(t, cens, side="right") - 1]

plt.figure(figsize=(6, 4))
plt.step(t, p, where="post")
plt.plot(cens, cens_prob, "|", color="black", markersize=8, label="censored")
plt.axhline(0.5, color="grey", linestyle=":")
plt.xlabel("days")
plt.ylabel("fraction surviving")
plt.title(f"Kaplan-Meier, overall survival ({RULE} rule)")
plt.legend()
plt.tight_layout()
plt.savefig(FIG + f"km_overall_{RULE}.png", dpi=150)
plt.show()

## 2. mRNA

Values are log2(RSEM + 1) expression, already normalized between samples by the source.

Three views:
- **All values:** a big spike near 0 means many gene-patient pairs with no detectable expression.
- **Per-gene mean:** genes whose mean is near 0 are barely expressed in leukemia blood cells and carry little signal.
- **Per-patient median:** one patient far from the rest may be a technical outlier.

In [ ]:
values = mrna.to_numpy().ravel()
gene_mean = mrna.mean(axis=1)
patient_median = mrna.median(axis=0) # to detect the outliers if exist

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].hist(values, bins=80)
ax[0].set_title("All mRNA values")
ax[0].set_xlabel("log2(RSEM + 1)")
ax[1].hist(gene_mean, bins=80)
ax[1].set_title("Per-gene mean")
ax[1].set_xlabel("mean log2(RSEM + 1)")
ax[2].hist(patient_median, bins=30)
ax[2].set_title("Per-patient median")
ax[2].set_xlabel("median log2(RSEM + 1)")
plt.tight_layout()
plt.savefig(FIG + f"mrna_distributions_{RULE}.png", dpi=200)
plt.show()

print("genes with mean < 1 (illustrative, not a filter yet):", int((gene_mean < 1).sum()),
      "of", len(gene_mean))
print("per-patient median: min", round(patient_median.min(), 2),
      "| max", round(patient_median.max(), 2)) #no patient has a global technical problem.

## 3. Methylation

Each value is a **beta**: the fraction of copies of that CpG site that are methylated in the sample (0 = none, 1 = all). Most CpGs sit near 0 or near 1, so the histogram should have two humps.

Beta is bounded, and its spread is squeezed near 0 and 1: a change from 0.02 to 0.05 and one from 0.50 to 0.53 look equal in beta but are very different in the underlying signal. The **M-value** `log2(beta / (1 - beta))` stretches the ends so that variability is more comparable across the range. We only look at it here; we decide later whether to model with it.

To keep plotting fast we sample 20,000 probes for the histograms.

In [ ]:
rng = np.random.default_rng(SEED)
sampled = rng.choice(meth.index, size=min(20000, len(meth)), replace=False)
beta = meth.loc[sampled].to_numpy().ravel() # bimodal, healthy 450k dataset

eps = 0.001                                   # keeps log2() finite at exactly 0 or 1
b = np.clip(beta, eps, 1 - eps)
mvalue = np.log2(b / (1 - b))

patient_mean_beta = meth.mean(axis=0)

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].hist(beta, bins=80)
ax[0].set_title("beta values (20,000 sampled probes)")
ax[0].set_xlabel("beta")
ax[1].hist(mvalue, bins=80)
ax[1].set_title("same values as M-values")
ax[1].set_xlabel("M-value")
ax[2].hist(patient_mean_beta, bins=30)
ax[2].set_title("per-patient mean beta (all probes)")
ax[2].set_xlabel("mean beta")
plt.tight_layout()
plt.savefig(FIG + f"methylation_distributions_{RULE}.png", dpi=200)
plt.show()

print("per-patient mean beta: min", round(patient_mean_beta.min(), 3),
      "| max", round(patient_mean_beta.max(), 3))
print(patient_mean_beta.sort_values(ascending=False).head(5).round(3))

## 4. What to write down

After running this, note here (and in the README) what you saw:
- Median survival and follow-up from the KM curve.
- Whether any patient looked like an outlier in either layer, and what you decided (keep or exclude, and why).
- Whether PCA showed clear groups, and in which layer.
- Anything that changes the plan, for example a clear technical split.

Then rerun with `RULE = "floor"` and check that nothing important changes.